# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Alien-is-here/FlyRank/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

**Answer:**

One feature row represents one client + one content item aggregated across the March 2026 development window.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


In [5]:
from google.colab import userdata
HF_TOKEN = userdata.get("HF_TOKEN")

print("HF_TOKEN loaded:", HF_TOKEN is not None)

HF_TOKEN loaded: True


In [6]:
import duckdb
import os
import pandas as pd
import numpy as np

con = duckdb.connect()

HF_TOKEN = userdata.get("HF_TOKEN")

con.execute(f"""
CREATE OR REPLACE SECRET hf_secret (
    TYPE HUGGINGFACE,
    TOKEN '{HF_TOKEN}'
)
""")

march_path = (
    "hf://datasets/FlyRank/internship-warehouse/"
    "fact_content_daily_performance/month=2026-03/*.parquet"
)


# One feature row = one client + one content item across the March 2026 development window.

query = f"""
SELECT
    client_hash_id AS client_id,
    content_hash_id AS content_id,

    SUM(gsc_impressions) AS gsc_impressions,
    SUM(gsc_clicks) AS gsc_clicks,

    AVG(NULLIF(gsc_avg_position, 0)) AS gsc_avg_position,

    SUM(
        CASE
            WHEN ga4_data_available IS TRUE
            THEN ga4_sessions
            ELSE NULL
        END
    ) AS ga4_sessions

FROM read_parquet('{march_path}')

GROUP BY
    client_hash_id,
    content_hash_id
"""

features = con.execute(query).df()



features["gsc_ctr"] = np.where(
    features["gsc_impressions"] > 0,
    features["gsc_clicks"] / features["gsc_impressions"],
    np.nan
)


feature_columns = [
    "gsc_impressions",
    "gsc_clicks",
    "gsc_avg_position",
    "ga4_sessions",
    "gsc_ctr"
]


#

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

 **Answer:**

### Selected features

The clustering features are:

* `gsc_impressions`
* `gsc_clicks`
* `gsc_avg_position`
* `ga4_sessions`
* `gsc_ctr`

`client_id` and `content_id` are identifiers/context fields, not clustering features.

No label is used because this is an exploratory, unsupervised clustering task.

### Excluded fields

Outcome-derived fields such as `trend_direction`, `trend_pct`, and `is_declining_label` are excluded because they would introduce outcome information into the clustering features.


In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

**Answer:**

I verify that:

1. Each `client_id + content_id` pair appears only once in the aggregated feature table.
2. The source data comes from the March 2026 partition.
3. The selected feature columns are checked for missing values.
4. GA4 availability is checked in the source data.


In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


In [13]:
print("\n=== GRAIN CHECK ===")

print("Total feature rows:", len(features))
print("Unique clients:", features["client_id"].nunique())
print("Unique contents:", features["content_id"].nunique())
print("Duplicate client + content pairs:",
      features.duplicated(["client_id", "content_id"]).sum())


=== GRAIN CHECK ===
Total feature rows: 331437
Unique clients: 55
Unique contents: 331437
Duplicate client + content pairs: 0


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

**Answer:**

The development data covers March 1–31, 2026 and contains 9,841,378 source rows.

GA4 availability is not consistent across the source data. There are 413,966 rows where GA4 data is available, 6,408,671 rows where it is unavailable, and 3,018,741 rows where GA4 availability is missing.

Therefore, GA4-based features may not represent the same amount of information for every observation. The resulting clusters should be interpreted as observed patterns in the available data, not as complete measurements of client or content performance.

The analysis is based on the March 2026 development window, so these patterns should not automatically be generalized to other time periods.


In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


In [10]:
features["gsc_impressions"].isnull().sum()

np.int64(0)

In [12]:
features[feature_columns].isnull().sum()

,0
gsc_impressions,0
gsc_clicks,0
gsc_avg_position,156133
ga4_sessions,240948
gsc_ctr,154699


In [15]:
print("========= Sourse Date Check ===============")
date_check = con.execute(f"""
    SELECT
        MIN(report_date) AS min_date,
        MAX(report_date) AS max_date,
        COUNT(*) AS sourse_rows
       FROM read_parquet('{march_path}')
        """).df()

print(date_check)

========= Sourse Date Check ===============


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

    min_date   max_date  sourse_rows
0 2026-03-01 2026-03-31      9841378


In [16]:
print("\n=== GA4 AVAILABILITY ===")

ga4_check = con.execute(f"""
    SELECT
        ga4_data_available,
        COUNT(*) AS rows
    FROM read_parquet('{march_path}')
    GROUP BY ga4_data_available
    ORDER BY ga4_data_available
""").df()

print(ga4_check)


=== GA4 AVAILABILITY ===
   ga4_data_available     rows
0               False  6408671
1                True   413966
2                <NA>  3018741


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.